In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity


# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

Current working directory:
E:\NLP_Project\intelligent-news-recommender\notebook

Project root:
E:\NLP_Project\intelligent-news-recommender


In [2]:
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print("Processed data folder:")
print(DATA_PROCESSED)

print("\nModels folder:")
print(MODELS_DIR)

print("\nFiles inside models folder:")

if MODELS_DIR.exists():
    for file in MODELS_DIR.iterdir():
        print(" -", file.name)
else:
    print("Models folder does not exist.")

Processed data folder:
E:\NLP_Project\intelligent-news-recommender\data\processed

Models folder:
E:\NLP_Project\intelligent-news-recommender\models

Files inside models folder:
 - tfidf_results.csv
 - tfidf_vectorizer.pkl


In [3]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root added to Python path.")

Project root added to Python path.


In [4]:
import importlib
import src.recommendation.recommender as recommender_module

# Force-reload the module in case the kernel already had an older
# cached version of NewsRecommender (e.g. before exclude_index was added)
importlib.reload(recommender_module)
from src.recommendation.recommender import NewsRecommender

print("NewsRecommender imported successfully.")

NewsRecommender imported successfully.


In [5]:
train_path = DATA_PROCESSED / "train_processed.csv"
test_path = DATA_PROCESSED / "test_processed.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (120000, 9)
Test shape : (7600, 6)


In [6]:
#Load processed datasets
train_path = DATA_PROCESSED / "train_processed.csv"
test_path = DATA_PROCESSED / "test_processed.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)


Train shape: (120000, 9)
Test shape : (7600, 6)


In [7]:
#Inspect the data
print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Train columns:
['label', 'title', 'description', 'category', 'text', 'word_count', 'clean_text', 'original_word_count', 'clean_word_count']

Test columns:
['label', 'title', 'description', 'category', 'text', 'clean_text']


In [8]:
#Check missing values
print("Missing values in test data:")
print(test_df.isnull().sum())

Missing values in test data:
label          0
title          0
description    0
category       0
text           0
clean_text     0
dtype: int64


In [9]:
# Load TF-IDF vectorizer

from pathlib import Path
import joblib

# Project root
PROJECT_ROOT = Path.cwd().parent

# Models folder
MODELS_DIR = PROJECT_ROOT / "models"

# TF-IDF vectorizer path
TFIDF_PATH = MODELS_DIR / "tfidf_vectorizer.pkl"

print("Current working directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nModels folder:")
print(MODELS_DIR)

print("\nTF-IDF vectorizer path:")
print(TFIDF_PATH)

# Check whether file exists
if not TFIDF_PATH.exists():
    print("\n❌ tfidf_vectorizer.pkl was NOT found.")
    print("\nFiles currently inside models folder:")

    if MODELS_DIR.exists():
        for file in MODELS_DIR.iterdir():
            print(" -", file.name)
    else:
        print("❌ Models folder does not exist.")

    raise FileNotFoundError(
        "\n\nTF-IDF vectorizer is missing."
        "\nGo to 03_tfidf_models.ipynb and save the vectorizer."
    )

# Load vectorizer
tfidf = joblib.load(TFIDF_PATH)

print("\n✅ TF-IDF vectorizer loaded successfully!")
print("Number of features:", len(tfidf.get_feature_names_out()))



Current working directory:
E:\NLP_Project\intelligent-news-recommender\notebook

Project root:
E:\NLP_Project\intelligent-news-recommender

Models folder:
E:\NLP_Project\intelligent-news-recommender\models

TF-IDF vectorizer path:
E:\NLP_Project\intelligent-news-recommender\models\tfidf_vectorizer.pkl

✅ TF-IDF vectorizer loaded successfully!
Number of features: 50000


In [10]:
#Inspect TF-IDF vectorizer
print("TF-IDF vocabulary size:")
print(len(tfidf.vocabulary_))

print("\nNumber of features:")
print(len(tfidf.get_feature_names_out()))

print("\nSample features:")
print(tfidf.get_feature_names_out()[:20])

TF-IDF vocabulary size:
50000

Number of features:
50000

Sample features:
['aa' 'aa billion' 'aaa' 'aapl' 'aaplo' 'aaron' 'aaron peirsol'
 'aaron rodgers' 'ab' 'ababa' 'abandon' 'abandon microsoft'
 'abandon nuclear' 'abandoned' 'abandoning' 'abarrel' 'abarrel mark'
 'abbas' 'abbey' 'abbey bid']


In [11]:
#Prepare recommendation dataset
news_data = test_df.copy()

print("News database shape:")
print(news_data.shape)

News database shape:
(7600, 6)


In [12]:
#Create recommendation engine
recommender = NewsRecommender(
    vectorizer=tfidf,
    news_data=news_data
)

print("News recommendation engine created successfully.")

News recommendation engine created successfully.


In [13]:
#Check TF-IDF matrix
recommender.tfidf_matrix
print("TF-IDF matrix shape:")
print(recommender.tfidf_matrix.shape)

TF-IDF matrix shape:
(7600, 50000)


In [14]:
#Select a sample article
article_index = 100

sample_article = test_df.iloc[article_index]

print("Article index:", article_index)

print("\nTitle:")
print(sample_article["title"])

print("\nDescription:")
print(sample_article["description"])

print("\nCategory:")
print(sample_article["label"])

Article index: 100

Title:
Olympic history for India, UAE

Description:
An Indian army major shot his way to his country #39;s first ever individual Olympic silver medal on Tuesday, while in the same event an member of Dubai #39;s ruling family became the first ever medallist from the United Arab Emirates. 

Category:
2


In [15]:
#Create query
query = sample_article["clean_text"]

print("Query:")
print(query)

Query:
olympic history india uae indian army major shot way country first ever individual olympic silver medal tuesday event member dubai ruling family became first ever medallist united arab emirate


In [16]:
#Get recommendations
recommendations = recommender.recommend(
    query=query,
    top_k=5,
    exclude_index=article_index
)

print("Recommendations generated successfully.")

Recommendations generated successfully.


In [17]:
#Display recommended articles
recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,United Arab Emirates trap shooter secures nati...,Sheik Ahmed bin Hashr Al-Maktoum earned the fi...,2,0.282397
1,Fired-up Baggaley takes silver,Australia #39;s Nathan Baggaley was over the m...,2,0.189137
2,Britain #39;s Olympic medal total takes sudden...,Great Britain #39;s performances in the Olympi...,2,0.170577
3,Boxing: Khan shows no rust to emerge lord of t...,Amir Khan celebrated his first fight since win...,2,0.158055
4,"UAE Founding Father Buried, Elder Son Likely S...","ABU DHABI, November 3 (IslamOnline.net amp; N...",1,0.150472


In [18]:
#Display recommendations more clearly
for i, row in recommendations.iterrows():

    print("=" * 80)
    print(f"Recommendation #{i + 1}")
    print("=" * 80)

    print("Title:", row["title"])
    print("Category:", row["label"])
    print("Similarity:", round(row["similarity_score"], 4))

    print("\nDescription:")
    print(row["description"])

    print()

Recommendation #1
Title: United Arab Emirates trap shooter secures nation #39;s first Olympic gold
Category: 2
Similarity: 0.2824

Description:
Sheik Ahmed bin Hashr Al-Maktoum earned the first-ever Olympic medal for the United Arab Emirates when he took home the gold medal in men #39;s double trap shooting on Tuesday in Athens. 

Recommendation #2
Title: Fired-up Baggaley takes silver
Category: 2
Similarity: 0.1891

Description:
Australia #39;s Nathan Baggaley was over the moon after winning the silver medal in the Olympic kayaking K1 500 event today. Double world champion Baggaley fired from the start and took an early lead but faded 

Recommendation #3
Title: Britain #39;s Olympic medal total takes sudden turn for the better
Category: 2
Similarity: 0.1706

Description:
Great Britain #39;s performances in the Olympic Games made a dramatic and unexpected improvement yesterday as they won a silver and three bronze medals. They were also guaranteed at least a silver medal in badminton #

In [19]:
#Test another article
article_index = 500

sample_article = test_df.iloc[article_index]

print("Original Article")
print("=" * 80)

print("Title:", sample_article["title"])
print("Category:", sample_article["label"])
print("Description:", sample_article["description"])

Original Article
Title: Iran shuts reformist websites
Category: 4
Description: WEBSITES CLOSE to Iran #39;s leading reformist party have been blocked by religious hardliners in the police bureau of public morals.


In [20]:
#Generate recommendations
query = sample_article["clean_text"]

recommendations = recommender.recommend(
    query=query,
    top_k=5,
    exclude_index=article_index
)

recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,Attack prompts Bush website block,The re-election website of President Bush is b...,4,0.158565
1,United Apology over Website Abuse,Manchester United have been forced to issue an...,2,0.140711
2,Communist Party seeks to win back people #39;s...,THE Chinese Communist Party (CCP) has read the...,1,0.116732
3,Europe compromises with US on Iran nuke deadline,charge Iran vehemently denies. The IAEA has fo...,1,0.116067
4,Freeze on anti-spam campaign,A campaign by Lycos Europe to target spam-rela...,4,0.112405


In [21]:
#Test with your own news query
custom_query = """
Apple announced new artificial intelligence technology
for its latest products, including advanced machine learning
features and improved performance.
"""

print("Custom query:")
print(custom_query)

Custom query:

Apple announced new artificial intelligence technology
for its latest products, including advanced machine learning
features and improved performance.



In [22]:
#Convert custom query to TF-IDF
custom_query_vector = tfidf.transform([custom_query])

print("Custom query vector shape:")
print(custom_query_vector.shape)

Custom query vector shape:
(1, 50000)


In [23]:
#Get recommendations for custom query
custom_recommendations = recommender.recommend(
    query=custom_query,
    top_k=5
)

custom_recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,Bill Clinton Helps Launch Search Engine,Former president Bill Clinton on Monday helped...,4,0.186304
1,Is Apple Photogenic?,With competitors avidly trying to nibble at th...,4,0.138930
2,2005 American Bowl teams announced,"New York, NY (Sports Network) - Indianapolis w...",2,0.127525
3,Regulators Approve Artificial Heart,The Food and Drug Administration approved the ...,4,0.125415
4,Google puts desktop search privacy up front,Google has announced a new desktop search appl...,3,0.119883


In [24]:
#Pretty display
for i, row in custom_recommendations.iterrows():

    print("=" * 80)
    print(f"Recommended News #{i + 1}")
    print("=" * 80)

    print("Title:", row["title"])
    print("Category:", row["label"])
    print("Similarity Score:", round(row["similarity_score"], 4))

    print("\nDescription:")
    print(row["description"])

    print()

Recommended News #1
Title: Bill Clinton Helps Launch Search Engine
Category: 4
Similarity Score: 0.1863

Description:
Former president Bill Clinton on Monday helped launch a new Internet search company backed by the Chinese government which says its technology uses artificial intelligence to produce better results than Google Inc.

Recommended News #2
Title: Is Apple Photogenic?
Category: 4
Similarity Score: 0.1389

Description:
With competitors avidly trying to nibble at the iPod #39;s market share, Apple (Nasdaq: AAPL) has released its ostensibly new and improved version.

Recommended News #3
Title: 2005 American Bowl teams announced
Category: 2
Similarity Score: 0.1275

Description:
New York, NY (Sports Network) - Indianapolis will take on Atlanta in the 2005 American Bowl in Japan, the league announced Friday.

Recommended News #4
Title: Regulators Approve Artificial Heart
Category: 4
Similarity Score: 0.1254

Description:
The Food and Drug Administration approved the use of an art

In [25]:
#Test different categories
sports_query = """
Manchester United won an important football match after
scoring two goals in the second half. The team is preparing
for its next Premier League game.
"""

sports_recommendations = recommender.recommend(
    query=sports_query,
    top_k=5
)

sports_recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,Manchester United Beats Charlton 2-0 (AP),AP - Manchester United defeated Charlton 2-0 S...,2,0.265801
1,Smith saves United,"LONDON, Aug. 28. - Alan Smith scored a late eq...",2,0.239422
2,Manchester United admits paying 11m to transfe...,The role of agents in multimillion-pound footb...,2,0.234222
3,"Manchester United the only team for me, says R...",Teenage striker Wayne Rooney says Manchester U...,2,0.232500
4,Manchester United cruise into Champions League,Manchester United eased into the Champions Lea...,2,0.225787


In [26]:
#Test Business query
business_query = """
The stock market rose after technology companies reported
strong quarterly earnings. Investors are expecting continued
growth in the technology sector.
"""

business_recommendations = recommender.recommend(
    query=business_query,
    top_k=5
)

business_recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,FedEx Quarterly Earnings More Than Double,The world's top air-express shipper said earni...,3,0.190881
1,Medtronic Quarterly Earnings Rise,CHICAGO (Reuters) - Medtronic Inc. &lt;A HREF...,3,0.179090
2,Lowe #39;s Q3 earnings jump 15.5 percent,NEW YORK (CBS.MW) - Lowe #39;s reported a stro...,3,0.170189
3,Update 3: ADM #39;s Earnings Skyrocket; Stocks...,Shares in agribusiness giant Archer Daniels Mi...,3,0.151701
4,Stocks Seen Flat as Earnings Pour In,US stock futures pointed to a flat market open...,3,0.146217


In [27]:
#Test World News query
world_query = """
The government announced a new international agreement
to improve cooperation between countries and strengthen
economic and diplomatic relations.
"""

world_recommendations = recommender.recommend(
    query=world_query,
    top_k=5
)

world_recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,UN Signs Pact with New World Court Opposed by ...,UNITED NATIONS (Reuters) - The United Nations...,1,0.235986
1,Cellphones outstrip landlines in India (AFP),AFP - Mobile phone users have outstripped trad...,4,0.194582
2,"China, Argentina sign 5 cooperation documents",China and Argentina signed five agreements in ...,3,0.145367
3,2005 American Bowl teams announced,"New York, NY (Sports Network) - Indianapolis w...",2,0.144690
4,Britain plans national ID cards,London -- Invoking a global threat of terroris...,1,0.116727


In [28]:
#Test Sci/Tech query
technology_query = """
Researchers developed a new artificial intelligence system
that can analyze medical images using deep learning and
improve disease detection.
"""

technology_recommendations = recommender.recommend(
    query=technology_query,
    top_k=5
)

technology_recommendations[
    [
        "title",
        "description",
        "label",
        "similarity_score"
    ]
]

,title,description,label,similarity_score
0,Bill Clinton Helps Launch Search Engine,Former president Bill Clinton on Monday helped...,4,0.153745
1,Obesity Solution: Nuke It,Eying that juicy steak but worried about your ...,4,0.139844
2,Regulators Approve Artificial Heart,The Food and Drug Administration approved the ...,4,0.131845
3,Yankees' Holding Pattern Is Sure to Change,"Brian Cashman, the general manager of the Yank...",2,0.108225
4,Mich. Rep. to Head Intelligence Panel (AP),AP - Republican Rep. Peter Hoekstra of Michiga...,1,0.096747


In [29]:
#Create a reusable function
def show_recommendations(query, top_k=5):

    recommendations = recommender.recommend(
        query=query,
        top_k=top_k
    )

    print("\n")
    print("=" * 80)
    print("NEWS RECOMMENDATIONS")
    print("=" * 80)

    for i, row in recommendations.iterrows():

        print(f"\nRecommendation #{i + 1}")
        print("-" * 80)

        print("Title:", row["title"])
        print("Category:", row["label"])
        print(
            "Similarity Score:",
            round(row["similarity_score"], 4)
        )

        print("\nDescription:")
        print(row["description"])

    return recommendations

In [30]:
#Use the reusable function
query = """
Tesla announced new electric vehicle technology
that improves battery performance and driving range.
"""

results = show_recommendations(
    query=query,
    top_k=5
)



NEWS RECOMMENDATIONS

Recommendation #1
--------------------------------------------------------------------------------
Title: Kyocera Battery Recall Kyocera Battery Recall
Category: 4
Similarity Score: 0.163

Description:
By guest contributor Josh Pereira. Kyocera, a leading manufacturer of CDMA phones, has announced a voluntary and precautionary recall of the batteries found in their KE/KX 400 Series, 3200 Series, and Slider Series phones.

Recommendation #2
--------------------------------------------------------------------------------
Title: 2005 American Bowl teams announced
Category: 2
Similarity Score: 0.1607

Description:
New York, NY (Sports Network) - Indianapolis will take on Atlanta in the 2005 American Bowl in Japan, the league announced Friday.

Recommendation #3
--------------------------------------------------------------------------------
Title: Hologram labels in Nokia batteries
Category: 4
Similarity Score: 0.1606

Description:
New Delhi: To help customers ident

In [31]:
#Save recommendation results
results_path = PROJECT_ROOT / "data" / "interim" / "recommendation_results.csv"

results.to_csv(
    results_path,
    index=False
)

print("Recommendation results saved to:")
print(results_path)

Recommendation results saved to:
E:\NLP_Project\intelligent-news-recommender\data\interim\recommendation_results.csv


In [32]:
#Check recommendation scores
results[
    [
        "title",
        "label",
        "similarity_score"
    ]
].sort_values(
    by="similarity_score",
    ascending=False
)

,title,label,similarity_score
0,Kyocera Battery Recall Kyocera Battery Recall,4,0.163028
1,2005 American Bowl teams announced,2,0.160722
2,Hologram labels in Nokia batteries,4,0.160564
3,Qatar Defense Show Focuses on Videogame Techno...,4,0.153153
4,Nokia To Use Holograms To Thwart Battery Count...,4,0.143973


In [33]:
#Verify recommendation order
scores = results["similarity_score"].values

print("Similarity scores:")
print(scores)

print("\nAre recommendations sorted by similarity?")

is_sorted = all(
    scores[i] >= scores[i + 1]
    for i in range(len(scores) - 1)
)

print(is_sorted)

Similarity scores:
[0.16302766 0.16072202 0.16056433 0.1531528  0.14397329]

Are recommendations sorted by similarity?
True


In [34]:
#Final project test
final_query = """
Google introduced a new artificial intelligence model
designed to improve search, language understanding,
and machine learning applications.
"""

final_results = show_recommendations(
    query=final_query,
    top_k=5
)



NEWS RECOMMENDATIONS

Recommendation #1
--------------------------------------------------------------------------------
Title: Bill Clinton Helps Launch Search Engine
Category: 4
Similarity Score: 0.2245

Description:
Former president Bill Clinton on Monday helped launch a new Internet search company backed by the Chinese government which says its technology uses artificial intelligence to produce better results than Google Inc.

Recommendation #2
--------------------------------------------------------------------------------
Title: Understanding Search Engine Models
Category: 4
Similarity Score: 0.2017

Description:
Understanding Search Engine Models\\To understand search engines and search engine marketing, one must first understand the search engine model. There are two fundamentally different types of search engine back ends: site directories and spidering search engines. Site directory databases are built by a person manually inputting data about websites. Most ...

Recommenda